# Rectified Flow → Reflow: весь код обучения в ноутбуке

Здесь обучаем **1-Rectified Flow с нуля**, замораживаем её, получаем связанные пары
`(initial noise, generated endpoint)` и обучаем **2-Rectified Flow** на прямых мостах между
этими парами. Модель, optimizer, random generators, оба training loops и Euler sampler видны ниже.
Готовые веса не скачиваются.

Идея reflow: повторно обучать velocity на coupling, полученном предыдущей ODE, чтобы сделать
траектории прямее и уменьшить ошибку при малом числе шагов.
Первоисточники: [Liu, Gong, Liu, *Flow Straight and Fast*, 2022 / ICLR 2023](https://arxiv.org/abs/2209.03003),
[объяснение авторов](https://www.cs.utexas.edu/~lqiang/rectflow/html/intro.html),
[официальный код](https://github.com/gnobitab/RectifiedFlow).

По умолчанию по 100 эпох на 4096 точках вращающихся лучей, четыре класса.
Уменьшение бюджета проверяет исполнение, но может скрыть пользу reflow; результат оцениваем по
held-out данным, endpoint fidelity и геометрии плотных ODE-траекторий.


In [ ]:
# Установка исходного пакета в текущее Jupyter / Colab ядро.
# Для воспроизводимого окружения замените main на commit SHA или tag.
import os
import subprocess
import sys

GIT_SOURCE = "diffusion-lab[interactive] @ git+https://github.com/Vovanm88/ddpm_vibe_research.git@main"
# Служебный флаг для проверки уже установленного локального checkout.
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", GIT_SOURCE])


In [ ]:
import copy
import hashlib
import importlib.metadata
import json
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import FlowMatchEulerDiscreteScheduler
from diffusers.utils.torch_utils import randn_tensor
from IPython.display import Markdown, display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from diffusion_lab import PointDenoiser, ResearchDiffusionPipeline
from diffusion_lab.environments import make_environment
from diffusion_lab.metrics import evaluate_samples
from diffusion_lab.visualization import (
    plot_dataset, plot_rainbow_trajectories, save_figure, save_trajectory_gif,
)


In [ ]:
SEED = 2026
DEVICE = "cpu"                    # "cuda" / "mps", если доступны
N_TRAIN = 4096
N_PAIRS = 4096
N_EVAL = 1024
N_CLASSES = 4
HIDDEN_DIM = 96
NUM_LAYERS = 3
BATCH_SIZE = 256
RF_EPOCHS = 100
REFLOW_EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CONDITION_DROPOUT = 0.15
GRAD_CLIP = 1.0
TRAIN_TIMESTEPS = 1000
TEACHER_STEPS = 64
DENSE_STEPS = 64
EVAL_STEPS = (1, 4, 16, 64)
EVAL_BATCH_SIZE = 256
WARM_START_REFLOW = True
ENABLE_THIRD_REFLOW = False
THIRD_REFLOW_EPOCHS = 100
EVALUATE_UNCONDITIONAL = True
SAVE_GIFS = True
GIF_PARTICLES = 16
RESUME_EXTRA_EPOCHS = 0             # >0: выполнить видимый пример продолжения обучения
OUTPUT_DIR = "outputs/reflow"


In [ ]:
torch.manual_seed(SEED)
torch.set_num_threads(4)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
started_at = time.perf_counter()

train_data = make_environment("rotating_rays", n_samples=N_TRAIN, n_classes=N_CLASSES, seed=SEED)
heldout_data = make_environment("rotating_rays", n_samples=N_EVAL, n_classes=N_CLASSES, seed=SEED + 1)
print("Train / held-out shapes:", train_data.samples.shape, heldout_data.samples.shape)
plot_dataset(train_data.samples, train_data.conditions, title="Training data · rotating rays")
plt.show()


## Конвенция времени и target

В статье время направлено от начального распределения к конечному. Здесь, как в
`FlowMatchEulerDiscreteScheduler`, **σ=1 — Gaussian noise, σ=0 — data**.
Переобозначение `t=1−σ` меняет знак velocity:

\[
x_\sigma=(1-\sigma)y+\sigma z,\qquad
v_\theta(x_\sigma,1000\sigma,c)\approx z-y.
\]

В 1-RF точки `y` из датасета и Gaussian `z` независимы.
Во 2-RF `y=T_1(z,c)` — endpoint замороженной 1-RF **для того же z**.
Euler идёт от σ=1 к 0, поэтому update имеет отрицательный Δσ.
Это линейный RF objective, реализующий [прямую интерполяцию авторов](https://www.cs.utexas.edu/~lqiang/rectflow/html/intro.html).


In [ ]:
model_1rf = PointDenoiser(
    sample_shape=train_data.sample_shape,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
    backbone="mlp", conditioner="embedding", num_classes=N_CLASSES,
).to(DEVICE)
optimizer_1rf = torch.optim.AdamW(model_1rf.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loader_generator_1rf = torch.Generator().manual_seed(SEED + 2)
training_generator_1rf = torch.Generator().manual_seed(SEED + 3)
loader_1rf = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                       generator=loader_generator_1rf, num_workers=0)
loss_fn = nn.MSELoss()


## 1. Обучаем 1-RF: обычный PyTorch loop

Каждый batch получает новый независимый Gaussian noise и случайный σ.
Conditioning dropout учит null-условие; класс не заменяется новым ID.
Ни sampler, ни teacher не участвуют в этом первом training loop.


In [ ]:
history_1rf = []
for epoch in range(RF_EPOCHS):
    model_1rf.train()
    sum_loss, count = 0.0, 0
    for batch in loader_1rf:
        data = batch["samples"].to(device=DEVICE, dtype=model_1rf.dtype)
        labels = batch["conditions"].to(DEVICE)
        noise = randn_tensor(data.shape, generator=training_generator_1rf,
                             device=model_1rf.device, dtype=model_1rf.dtype)
        sigma = torch.rand(len(data), generator=training_generator_1rf).to(DEVICE)
        expanded_sigma = sigma.reshape(-1, *([1] * (data.ndim - 1)))
        bridge = (1 - expanded_sigma) * data + expanded_sigma * noise
        target_velocity = noise - data
        keep = (torch.rand(len(data), generator=training_generator_1rf) >= CONDITION_DROPOUT).to(DEVICE)

        optimizer_1rf.zero_grad(set_to_none=True)
        predicted_velocity = model_1rf(
            bridge, sigma * TRAIN_TIMESTEPS, conditions=labels, condition_mask=keep,
        ).sample
        loss = loss_fn(predicted_velocity, target_velocity)
        if not torch.isfinite(loss):
            raise FloatingPointError("1-RF loss is not finite")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_1rf.parameters(), GRAD_CLIP, error_if_nonfinite=True)
        optimizer_1rf.step()
        sum_loss += loss.detach().item() * len(data)
        count += len(data)
    history_1rf.append({"epoch": epoch + 1, "loss": sum_loss / count,
                        "optimizer_steps": (epoch + 1) * len(loader_1rf)})
    if epoch == 0 or (epoch + 1) % 25 == 0 or epoch + 1 == RF_EPOCHS:
        print(f"1-RF epoch {epoch + 1}/{RF_EPOCHS}: loss={history_1rf[-1]['loss']:.5f}", flush=True)


## 2. Видимый Euler sampler, исходные latents задаём сами

Каждый вызов создаёт новый официальный scheduler без mutable state от предыдущей генерации.
`condition_mask` фиксирован для частицы на **всех** шагах ODE. При `conditions=None`
используется null-условие. `initial_noise` не меняется: работаем с его копией.


In [ ]:
@torch.no_grad()
def sample_euler(model, initial_noise, conditions=None, condition_mask=None,
                 steps=64, batch_size=256, record_history=True):
    if not 1 <= steps <= TRAIN_TIMESTEPS:
        raise ValueError("steps must be in [1, TRAIN_TIMESTEPS]")
    if len(initial_noise) < 1 or tuple(initial_noise.shape[1:]) != model.sample_shape:
        raise ValueError("initial_noise must be [N,*model.sample_shape]")
    if conditions is not None and len(conditions) != len(initial_noise):
        raise ValueError("condition and noise batch sizes must match")
    if condition_mask is not None and len(condition_mask) != len(initial_noise):
        raise ValueError("mask and noise batch sizes must match")
    final_batches, history_batches = [], []
    was_training = model.training
    model.eval()
    try:
        for start in range(0, len(initial_noise), batch_size):
            stop = start + batch_size
            state = initial_noise[start:stop].to(device=model.device, dtype=model.dtype).clone()
            labels = conditions[start:stop].to(model.device) if conditions is not None else None
            keep = condition_mask[start:stop].to(model.device) if condition_mask is not None else None
            scheduler = FlowMatchEulerDiscreteScheduler(num_train_timesteps=TRAIN_TIMESTEPS, shift=1.0)
            scheduler.set_timesteps(steps, device=model.device)
            states = [state.detach().cpu().clone()] if record_history else None
            for timestep in scheduler.timesteps:
                velocity = model(state, timestep, conditions=labels, condition_mask=keep).sample
                state = scheduler.step(velocity, timestep, state).prev_sample
                if record_history:
                    states.append(state.detach().cpu().clone())
            final_batches.append(state.detach().cpu())
            if record_history:
                history_batches.append(torch.stack(states))
    finally:
        model.train(was_training)
    return {
        "samples": torch.cat(final_batches),
        "history": torch.cat(history_batches, dim=1) if record_history else None,
        "sigmas": scheduler.sigmas.detach().cpu().clone(),
        "timesteps": scheduler.timesteps.detach().cpu().clone(),
    }


## 3. Замораживаем teacher и сохраняем coupling

В [официальном workflow](https://github.com/gnobitab/RectifiedFlow) сначала генерируются пары,
затем модель дообучается на них. У каждой строки ниже сохраняются Gaussian noise, endpoint,
label и один keep-mask. Даже null endpoint получен teacher именно с null-mask.
Shuffle допустим только для **целых строк**, а не независимо для двух концов пары.


In [ ]:
model_1rf.eval().requires_grad_(False)
teacher_before = {name: value.detach().cpu().clone() for name, value in model_1rf.state_dict().items()}
pair_generator = torch.Generator().manual_seed(SEED + 10)
pair_noise = torch.randn((N_PAIRS, *train_data.sample_shape), generator=pair_generator)
pair_labels = torch.arange(N_PAIRS, dtype=torch.long) % N_CLASSES
pair_keep = torch.rand(N_PAIRS, generator=pair_generator) >= CONDITION_DROPOUT

teacher_pairs = sample_euler(model_1rf, pair_noise, pair_labels, pair_keep,
                             steps=TEACHER_STEPS, batch_size=EVAL_BATCH_SIZE, record_history=False)
pair_endpoints = teacher_pairs["samples"].detach().clone()
assert not pair_endpoints.requires_grad
assert not any(parameter.requires_grad for parameter in model_1rf.parameters())
paired_dataset = TensorDataset(pair_noise, pair_endpoints, pair_labels, pair_keep)
# Каждый dataset[index] содержит оба связанных конца одной teacher-траектории.
torch.save({"noise": pair_noise, "endpoint": pair_endpoints,
            "labels": pair_labels, "keep_mask": pair_keep,
            "generation_seed": SEED + 10, "teacher_steps": TEACHER_STEPS}, output_dir / "pairs_1rf.pt")
print("Generated fixed pairs:", len(paired_dataset), "null-mask fraction:", float((~pair_keep).float().mean()))


## 4. Обучаем 2-RF на связанных концах — полный loop

Инициализация по умолчанию — копия обученной 1-RF, как fine-tuning в официальном коде.
`WARM_START_REFLOW=False` создаёт новую модель. Teacher остаётся отдельным frozen объектом.
Noise внутри каждой пары уже фиксирован; заново выбирается только σ.
**Сохранённый keep-mask повторяется при обучении**, чтобы null не получил class-generated target.


In [ ]:
def train_reflow(model, paired_loader, optimizer, epochs, generator, *, initial_epoch=0, stage="2-RF"):
    history = []
    for epoch in range(initial_epoch, initial_epoch + epochs):
        model.train()
        sum_loss, count = 0.0, 0
        for original_noise, generated_endpoint, labels, saved_keep in paired_loader:
            noise = original_noise.to(device=model.device, dtype=model.dtype)
            endpoint = generated_endpoint.to(device=model.device, dtype=model.dtype)
            labels, keep = labels.to(model.device), saved_keep.to(model.device)
            sigma = torch.rand(len(noise), generator=generator).to(model.device)
            expanded_sigma = sigma.reshape(-1, *([1] * (noise.ndim - 1)))
            straight_bridge = (1 - expanded_sigma) * endpoint + expanded_sigma * noise
            target_velocity = noise - endpoint

            optimizer.zero_grad(set_to_none=True)
            prediction = model(straight_bridge, sigma * TRAIN_TIMESTEPS,
                               conditions=labels, condition_mask=keep).sample
            loss = loss_fn(prediction, target_velocity)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"{stage} loss is not finite")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP, error_if_nonfinite=True)
            optimizer.step()
            sum_loss += loss.detach().item() * len(noise)
            count += len(noise)
        history.append({"epoch": epoch + 1, "loss": sum_loss / count,
                        "optimizer_steps": (epoch + 1) * len(paired_loader)})
        if epoch == initial_epoch or (epoch + 1) % 25 == 0 or epoch + 1 == initial_epoch + epochs:
            print(f"{stage} epoch {epoch + 1}/{initial_epoch + epochs}: loss={history[-1]['loss']:.5f}", flush=True)
    return history

if WARM_START_REFLOW:
    model_2rf = copy.deepcopy(model_1rf).requires_grad_(True).to(DEVICE)
else:
    torch.manual_seed(SEED + 11)
    model_2rf = PointDenoiser(sample_shape=train_data.sample_shape,
                             hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                             conditioner="embedding", num_classes=N_CLASSES).to(DEVICE)
optimizer_2rf = torch.optim.AdamW(model_2rf.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loader_generator_2rf = torch.Generator().manual_seed(SEED + 12)
training_generator_2rf = torch.Generator().manual_seed(SEED + 13)
loader_2rf = DataLoader(paired_dataset, batch_size=BATCH_SIZE, shuffle=True,
                       generator=loader_generator_2rf, num_workers=0)
history_2rf = train_reflow(model_2rf, loader_2rf, optimizer_2rf,
                          REFLOW_EPOCHS, training_generator_2rf)

assert model_2rf is not model_1rf
for name, value in model_1rf.state_dict().items():
    torch.testing.assert_close(value.detach().cpu(), teacher_before[name], rtol=0, atol=0)
print("Frozen 1-RF teacher is unchanged")
stages = {"1-RF": model_1rf, "2-RF": model_2rf}
training_histories = {"1-RF": history_1rf, "2-RF": history_2rf}


## 5. Необязательная третья rectification

По умолчанию отключена. Если включить, пары генерирует уже замороженная копия 2-RF,
после чего видимый `train_reflow` обучает 3-RF с теми же правилами pairing и mask.


In [ ]:
if ENABLE_THIRD_REFLOW:
    teacher_2rf = copy.deepcopy(model_2rf).eval().requires_grad_(False)
    third_pair_generator = torch.Generator().manual_seed(SEED + 20)
    third_noise = torch.randn((N_PAIRS, *train_data.sample_shape), generator=third_pair_generator)
    third_labels = torch.arange(N_PAIRS, dtype=torch.long) % N_CLASSES
    third_keep = torch.rand(N_PAIRS, generator=third_pair_generator) >= CONDITION_DROPOUT
    third_endpoint = sample_euler(teacher_2rf, third_noise, third_labels, third_keep,
                                  steps=TEACHER_STEPS, batch_size=EVAL_BATCH_SIZE,
                                  record_history=False)["samples"]
    third_dataset = TensorDataset(third_noise, third_endpoint, third_labels, third_keep)
    loader_generator_3rf = torch.Generator().manual_seed(SEED + 21)
    training_generator_3rf = torch.Generator().manual_seed(SEED + 22)
    loader_3rf = DataLoader(third_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           generator=loader_generator_3rf, num_workers=0)
    model_3rf = copy.deepcopy(teacher_2rf).requires_grad_(True).to(DEVICE)
    optimizer_3rf = torch.optim.AdamW(model_3rf.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history_3rf = train_reflow(model_3rf, loader_3rf, optimizer_3rf,
                              THIRD_REFLOW_EPOCHS, training_generator_3rf, stage="3-RF")
    torch.save({"noise": third_noise, "endpoint": third_endpoint, "labels": third_labels,
                "keep_mask": third_keep, "generation_seed": SEED + 20,
                "teacher_steps": TEACHER_STEPS}, output_dir / "pairs_2rf.pt")
    stages["3-RF"], training_histories["3-RF"] = model_3rf, history_3rf


## 6. Held-out сравнение: качество, fidelity, геометрия

Для каждой модели и числа шагов используем **одни held-out latents и labels**.
`SWD` сравнивает распределение с новым real-data датасетом; считаем также каждый класс.
`Teacher endpoint L2` сравнивает точку с результатом 1-RF на тех же latent / condition.

Длина/chord считается на отдельной плотной 64-step ODE: 1 означает прямой путь.
У одноступенчатого sampler геометрия всегда тривиально прямая, поэтому её не используем
как доказательство straightening. Ещё считаем velocity deviation вдоль плотной траектории.
Меньший reflow loss сам по себе не показывает улучшение SWD.


In [ ]:
def dense_geometry(sample):
    # Flatten только sample coordinates; batch и time остаются отдельными.
    trajectory = sample["history"].double().flatten(2)
    displacement = trajectory[1:] - trajectory[:-1]
    path_length = displacement.norm(dim=-1).sum(0)
    chord = (trajectory[-1] - trajectory[0]).norm(dim=-1)
    valid = chord > 1e-8
    delta_sigma = sample["sigmas"].double().diff()
    velocity = displacement / delta_sigma[:, None, None]
    chord_velocity = trajectory[0] - trajectory[-1]  # Δx / Δσ при σ: 1 → 0
    velocity_deviation = ((velocity - chord_velocity[None]).square().sum(-1)
                          * delta_sigma.abs()[:, None]).sum(0)
    return {
        "dense_length_over_chord": float((path_length[valid] / chord[valid]).mean()) if valid.any() else 1.0,
        "dense_velocity_deviation": float(velocity_deviation.mean()),
        "nonzero_chord_fraction": float(valid.double().mean()),
    }

heldout_generator = torch.Generator().manual_seed(SEED + 30)
eval_noise = torch.randn((N_EVAL, *train_data.sample_shape), generator=heldout_generator)
eval_labels = heldout_data.conditions.clone()
teacher_reference = sample_euler(model_1rf, eval_noise, eval_labels,
                                  steps=TEACHER_STEPS, batch_size=EVAL_BATCH_SIZE,
                                  record_history=False)["samples"]
comparison_rows, evaluation_outputs, dense_outputs = [], {}, {}
for stage_name, stage_model in stages.items():
    dense = sample_euler(stage_model, eval_noise, eval_labels,
                         steps=DENSE_STEPS, batch_size=EVAL_BATCH_SIZE)
    dense_outputs[stage_name] = dense
    geometry = dense_geometry(dense)
    for steps in EVAL_STEPS:
        sampled = dense if steps == DENSE_STEPS else sample_euler(
            stage_model, eval_noise, eval_labels, steps=steps,
            batch_size=EVAL_BATCH_SIZE, record_history=False)
        evaluation_outputs[(stage_name, steps)] = sampled["samples"]
        distribution = evaluate_samples(heldout_data, sampled["samples"], eval_labels, seed=SEED + 31)
        endpoint_error = (sampled["samples"] - teacher_reference).flatten(1).norm(dim=1).mean().item()
        comparison_rows.append({"stage": stage_name, "steps": int(steps),
                                **distribution, **geometry, "teacher_endpoint_l2": endpoint_error})

rows = ["| Stage | Steps | SWD ↓ | Teacher endpoint L2 ↓ | Dense length/chord ↓ | Dense velocity deviation ↓ |",
        "|---|---:|---:|---:|---:|---:|"]
for row in comparison_rows:
    rows.append(f"| {row['stage']} | {row['steps']} | {row['sliced_wasserstein']:.4f} | "
                f"{row['teacher_endpoint_l2']:.4f} | {row['dense_length_over_chord']:.4f} | "
                f"{row['dense_velocity_deviation']:.4f} |")
display(Markdown("\n".join(rows)))

class_rows = ["| Stage / steps | " + " | ".join(f"Class {i} SWD ↓" for i in range(N_CLASSES)) + " |",
              "|---|" + "---:|" * N_CLASSES]
for row in comparison_rows:
    values = [f"{row[f'class_{i}_sliced_wasserstein']:.4f}" for i in range(N_CLASSES)]
    class_rows.append(f"| {row['stage']} / {row['steps']} | " + " | ".join(values) + " |")
display(Markdown("\n".join(class_rows)))
(output_dir / "comparison.json").write_text(json.dumps(comparison_rows, indent=2, allow_nan=False) + "\n")


In [ ]:
fig, axes = plt.subplots(len(stages), len(EVAL_STEPS) + 1,
                         figsize=(3.1 * (len(EVAL_STEPS) + 1), 3.2 * len(stages)),
                         squeeze=False, constrained_layout=True)
for row_index, stage_name in enumerate(stages):
    plot_dataset(heldout_data.samples, heldout_data.conditions,
                  ax=axes[row_index, 0], title=f"Held-out reference · {stage_name}")
    for col_index, steps in enumerate(EVAL_STEPS, start=1):
        plot_dataset(evaluation_outputs[(stage_name, steps)], eval_labels,
                      ax=axes[row_index, col_index], title=f"{stage_name} · Euler N={steps}")
# Общие physical limits и одна легенда вне samples, чтобы сравнение было читаемым.
all_samples = torch.cat([heldout_data.samples, *evaluation_outputs.values()]).numpy()
lo, hi = all_samples.min(0), all_samples.max(0)
padding = np.maximum((hi - lo) * 0.08, 0.08)
legend_handles, legend_labels = axes[0, 0].get_legend_handles_labels()
for ax in axes.ravel():
    ax.set_xlim(lo[0] - padding[0], hi[0] + padding[0])
    ax.set_ylim(lo[1] - padding[1], hi[1] + padding[1])
    if ax.get_legend() is not None:
        ax.get_legend().remove()
fig.legend(legend_handles, legend_labels, loc="outside lower center", ncols=N_CLASSES,
           title="Condition", frameon=False, markerscale=1.4)
save_figure(fig, output_dir / "sample_comparison")
plt.show()

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
for name, history in training_histories.items():
    ax.plot([row["epoch"] for row in history], [row["loss"] for row in history], label=name)
ax.set(xlabel="Epoch", ylabel="Velocity MSE", title="Training losses: разные pair distributions")
ax.legend(frameon=False)
save_figure(fig, output_dir / "training_losses")
plt.show()


## 7. Null-условие: отдельный held-out coupling

Для unconditional teacher/reference берём новые latents и `conditions=None`.
Student null обучен на endpoints teacher с сохранённым false-mask, поэтому сравнение
teacher fidelity имеет тот же смысл, что в conditional случае.


In [ ]:
unconditional_rows = []
if EVALUATE_UNCONDITIONAL:
    null_noise = torch.randn((N_EVAL, *train_data.sample_shape),
                             generator=torch.Generator().manual_seed(SEED + 32))
    null_teacher_reference = sample_euler(model_1rf, null_noise, conditions=None,
                                          steps=TEACHER_STEPS, batch_size=EVAL_BATCH_SIZE,
                                          record_history=False)["samples"]
    for stage_name, stage_model in stages.items():
        dense_null = sample_euler(stage_model, null_noise, conditions=None,
                                  steps=DENSE_STEPS, batch_size=EVAL_BATCH_SIZE)
        null_geometry = dense_geometry(dense_null)
        for steps in EVAL_STEPS:
            sampled = dense_null if steps == DENSE_STEPS else sample_euler(
                stage_model, null_noise, conditions=None, steps=steps,
                batch_size=EVAL_BATCH_SIZE, record_history=False)
            distribution = evaluate_samples(heldout_data, sampled["samples"], conditions=None, seed=SEED + 31)
            endpoint_error = (sampled["samples"] - null_teacher_reference).flatten(1).norm(dim=1).mean().item()
            unconditional_rows.append({"stage": stage_name, "steps": int(steps), **distribution,
                                       **null_geometry, "teacher_endpoint_l2": endpoint_error})
    null_rows = ["| Null stage | Steps | SWD ↓ | Teacher endpoint L2 ↓ | Dense length/chord ↓ |",
                 "|---|---:|---:|---:|---:|"]
    for row in unconditional_rows:
        null_rows.append(f"| {row['stage']} | {row['steps']} | {row['sliced_wasserstein']:.4f} | "
                         f"{row['teacher_endpoint_l2']:.4f} | {row['dense_length_over_chord']:.4f} |")
    display(Markdown("\n".join(null_rows)))
    (output_dir / "unconditional_comparison.json").write_text(
        json.dumps(unconditional_rows, indent=2, allow_nan=False) + "\n")


## 8. Траектории: actual ODE, общие начальные точки

Радуга: красный Gaussian старт → оранжевый → жёлтый → зелёный → синий → фиолетовый endpoint.
Это реальные состояния 64-step Euler; прямые training bridges не выдаются за траектории ODE.
Все модели начинают с одинаковых latent, а оси для сравнения фиксируем общими.


In [ ]:
particle_count = min(GIF_PARTICLES, N_EVAL)
fig, axes = plt.subplots(1, len(stages), figsize=(6 * len(stages), 5.8),
                         squeeze=False, constrained_layout=True)
all_coordinates = torch.cat([sample["history"][:, :particle_count].reshape(-1, train_data.sample_shape[-1])
                              for sample in dense_outputs.values()]).numpy()
lower, upper = all_coordinates.min(0), all_coordinates.max(0)
margin = np.maximum((upper - lower) * 0.1, 0.08)
for ax, (stage_name, sampled) in zip(axes[0], dense_outputs.items()):
    paths = sampled["history"][:, :particle_count].permute(1, 0, 2)
    plot_rainbow_trajectories(paths, ax=ax, title=f"{stage_name} · actual {DENSE_STEPS}-step ODE",
                              start_label="Noise z", end_label="Sample y")
    ax.set_xlim(lower[0] - margin[0], upper[0] + margin[0])
    ax.set_ylim(lower[1] - margin[1], upper[1] + margin[1])
    if SAVE_GIFS:
        save_trajectory_gif(paths, output_dir / f"{stage_name.lower().replace('-', '')}_trajectories.gif",
                            fps=12, max_frames=50, title=f"{stage_name} · actual ODE, noise → sample",
                            start_label="Noise z", end_label="Sample y")
save_figure(fig, output_dir / "trajectory_comparison")
plt.show()
np.savez_compressed(output_dir / "evaluation_paths.npz", initial_noise=eval_noise.numpy(),
                    conditions=eval_labels.numpy(), teacher_reference=teacher_reference.numpy(),
                    **{f"{name.replace('-', '')}_history": value["history"].numpy()
                       for name, value in dense_outputs.items()})


## 9. Checkpoints: model, optimizer, данные, RNG и manifest

Diffusers сохраняет model config / weights / scheduler. PyTorch checkpoint хранит optimizer,
epoch history и состояния генераторов training / DataLoader. Teacher digest позволяет проверить,
что 1-RF не изменился при reflow. Исходные пары уже сохранены вместе с keep-mask.


In [ ]:
def model_digest(model):
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def save_stage(name, model, optimizer, history, training_generator, loader_generator):
    directory = output_dir / name
    pipeline = ResearchDiffusionPipeline(
        model, FlowMatchEulerDiscreteScheduler(num_train_timesteps=TRAIN_TIMESTEPS, shift=1.0))
    pipeline.save_pretrained(directory / "pipeline", safe_serialization=True)
    torch.save({"optimizer": optimizer.state_dict(), "completed_epochs": history[-1]["epoch"],
                "history": history, "training_generator": training_generator.get_state(),
                "loader_generator": loader_generator.get_state(), "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []},
               directory / "training.pt")
    return directory

save_stage("1rf", model_1rf, optimizer_1rf, history_1rf,
            training_generator_1rf, loader_generator_1rf)
save_stage("2rf", model_2rf, optimizer_2rf, history_2rf,
            training_generator_2rf, loader_generator_2rf)
if ENABLE_THIRD_REFLOW:
    save_stage("3rf", model_3rf, optimizer_3rf, history_3rf,
                training_generator_3rf, loader_generator_3rf)
torch.save({"samples": train_data.samples, "conditions": train_data.conditions,
            "heldout_samples": heldout_data.samples, "heldout_conditions": heldout_data.conditions},
           output_dir / "datasets.pt")

for name, value in model_1rf.state_dict().items():
    torch.testing.assert_close(value.detach().cpu(), teacher_before[name], rtol=0, atol=0)
parameters = {"seed": SEED, "device": DEVICE, "n_train": N_TRAIN, "n_pairs": N_PAIRS, "n_eval": N_EVAL,
              "n_classes": N_CLASSES, "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS,
              "batch_size": BATCH_SIZE, "rf_epochs": RF_EPOCHS, "reflow_epochs": REFLOW_EPOCHS,
              "learning_rate": LEARNING_RATE, "condition_dropout": CONDITION_DROPOUT,
              "teacher_steps": TEACHER_STEPS, "dense_steps": DENSE_STEPS, "eval_steps": list(EVAL_STEPS),
              "train_timesteps": TRAIN_TIMESTEPS, "weight_decay": WEIGHT_DECAY, "grad_clip": GRAD_CLIP,
              "eval_batch_size": EVAL_BATCH_SIZE, "third_reflow_epochs": THIRD_REFLOW_EPOCHS,
              "evaluate_unconditional": EVALUATE_UNCONDITIONAL, "save_gifs": SAVE_GIFS,
              "gif_particles": GIF_PARTICLES, "resume_extra_epochs": RESUME_EXTRA_EPOCHS,
              "warm_start_reflow": WARM_START_REFLOW, "third_reflow": ENABLE_THIRD_REFLOW}
manifest = {
    "algorithm": "1-Rectified Flow trained here, then paired teacher-endpoint reflow to 2-RF",
    "paper": "https://arxiv.org/abs/2209.03003",
    "time_convention": "sigma=1 noise; sigma=0 data; target velocity=noise-endpoint",
    "coupling": "whole-row noise/endpoint/label/fixed_keep_mask; teacher frozen during generation",
    "parameters": parameters, "teacher_sha256": model_digest(model_1rf),
    "python": platform.python_version(),
    "versions": {name: importlib.metadata.version(name) for name in ("torch", "diffusers", "diffusion-lab")},
    "duration_seconds": time.perf_counter() - started_at,
}
(output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, allow_nan=False) + "\n")
print("Saved artifacts:", output_dir.resolve())
print(f"Elapsed: {manifest['duration_seconds']:.1f} seconds")


## 10. Видимый пример restore / resume

`RESUME_EXTRA_EPOCHS=0` только восстанавливает checkpoint. Положительное значение продолжает
paired reflow с тем же optimizer, порядком loader и training RNG.
Для проверки восстановленной модели нужен исходный `PointDenoiser`; custom classes определите
перед loading. Используем собственные доверенные файлы и `weights_only=True`.


In [ ]:
restored_pipeline = ResearchDiffusionPipeline.from_pretrained(
    output_dir / "2rf" / "pipeline", local_files_only=True, low_cpu_mem_usage=False).to(DEVICE)
restored_model = restored_pipeline.model.requires_grad_(True)
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
resume_state = torch.load(output_dir / "2rf" / "training.pt", map_location="cpu", weights_only=True)
restored_optimizer.load_state_dict(resume_state["optimizer"])
restored_training_generator = torch.Generator().set_state(resume_state["training_generator"])
restored_loader_generator = torch.Generator().set_state(resume_state["loader_generator"])
pair_state = torch.load(output_dir / "pairs_1rf.pt", map_location="cpu", weights_only=True)
restored_dataset = TensorDataset(pair_state["noise"], pair_state["endpoint"],
                                 pair_state["labels"], pair_state["keep_mask"])
restored_loader = DataLoader(restored_dataset, batch_size=BATCH_SIZE, shuffle=True,
                             generator=restored_loader_generator, num_workers=0)
torch.set_rng_state(resume_state["torch_rng"])
if torch.cuda.is_available() and resume_state["cuda_rng"]:
    torch.cuda.set_rng_state_all(resume_state["cuda_rng"])
restored_sample = sample_euler(restored_model, eval_noise, eval_labels,
                               steps=DENSE_STEPS, batch_size=EVAL_BATCH_SIZE, record_history=False)["samples"]
torch.testing.assert_close(restored_sample, dense_outputs["2-RF"]["samples"], rtol=1e-5, atol=1e-6)
if RESUME_EXTRA_EPOCHS:
    resumed_history = train_reflow(restored_model, restored_loader, restored_optimizer,
                                   RESUME_EXTRA_EPOCHS, restored_training_generator,
                                   initial_epoch=resume_state["completed_epochs"], stage="2-RF resumed")
    all_history = resume_state["history"] + resumed_history
    save_stage("2rf_resumed", restored_model, restored_optimizer, all_history,
                restored_training_generator, restored_loader_generator)
print("Restore checked; optional extra epochs:", RESUME_EXTRA_EPOCHS)


Reflow здесь сохраняет coupling **приближённого teacher**: он не устраняет автоматически ошибки
1-RF относительно real data. Конечная модель, конечный датасет и Euler discretization могут дать
другой результат, чем идеализированная теория. Оценивайте совместно SWD, поклассовые метрики,
fidelity и плотную геометрию; повторяйте несколько seed.

Что менять дальше: `HIDDEN_DIM` / `NUM_LAYERS`, embedding → иной conditioner, `loss_fn`, optimizer,
распределение σ, число teacher steps и размер paired dataset. При замене задачи обновите её
`sample_shape` и conditioner sizes; показанный Euler sampler сохраняет произвольную форму samples.


## 11. Интерактивные поля и sampling обученной 2-RF

Эти панели используют уже обученную `model_2rf`; обучение повторно не запускается.
В field explorer меняйте класс / **Unconditional**, σ через timestep slider, вид поля,
стрелки / streamlines и восстановление потенциала.

**Velocity** — `dx/dσ`; генерация движется от σ=1 к σ=0.
**Denoising = −velocity** показывает направление noise → data.
Потенциал восстанавливается как приближённая проекция `F ≈ −∇U`: learned RF field
может иметь curl, поэтому проверяйте residual / curl в панели.
Null-условие обучено через dropout в 1-RF и фиксированные null-pairs при reflow.

Sampling начинается с **4 Euler steps**. Slider позволяет сравнить 1 / 4 / 16 / 64
и другие бюджеты; изменения применяются после **Generate**.
**Use CFG** по умолчанию выключен (`guidance_scale=1`), его можно включить отдельно.
**Play / Frame** воспроизводят реальные сохранённые состояния сэмплирования.

Для запуска без UI установите `ENABLE_INTERACTIVE_EXPLORERS=False` перед этими ячейками.


In [ ]:
ENABLE_INTERACTIVE_EXPLORERS = globals().get("ENABLE_INTERACTIVE_EXPLORERS", True)

if ENABLE_INTERACTIVE_EXPLORERS:
    # Нужно только Google Colab; обычный Jupyter использует свой widget renderer.
    try:
        from google.colab import output as colab_output
    except ImportError:
        pass
    else:
        colab_output.enable_custom_widget_manager()

    from diffusion_lab.visualization import notebook_field_explorer, notebook_sampling_explorer

    rf_live_pipeline = ResearchDiffusionPipeline(
        model_2rf,
        FlowMatchEulerDiscreteScheduler(num_train_timesteps=TRAIN_TIMESTEPS, shift=1.0),
    ).to(DEVICE)
    rf_field_explorer = notebook_field_explorer(
        rf_live_pipeline, dataset=train_data, resolution=24,
        timestep=TRAIN_TIMESTEPS // 2, guidance_scale=1.0,
    )
    rf_field_explorer.controls["cfg_enabled"].value = False
    if len(rf_field_explorer.controls["condition"].options) > 1:
        rf_field_explorer.controls["condition"].index = 1   # Первый класс; Unconditional тоже доступен.
    display(rf_field_explorer)


In [ ]:
if ENABLE_INTERACTIVE_EXPLORERS:
    rf_sampling_explorer = notebook_sampling_explorer(
        rf_live_pipeline, dataset=train_data, num_samples=128,
        steps=4, seed=SEED + 40, guidance_scale=1.0,
    )
    rf_sampling_explorer.controls["cfg_enabled"].value = False
    if len(rf_sampling_explorer.controls["condition"].options) > 1:
        rf_sampling_explorer.controls["condition"].index = 1
    rf_sampling_explorer.generate()        # Первый настоящий 4-step run; затем кнопка Generate.
    display(rf_sampling_explorer)
